Purpose: add lagged/rolling time-series features to the preprocessed Air Quality data for downstream models.

Load libraries and preprocessed data.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

artifacts = Path('artifacts')
artifacts.mkdir(exist_ok=True)

data_path = Path('preprocessed_air_quality.csv')
if not data_path.exists():
    raise FileNotFoundError('Missing preprocessed_air_quality.csv - run preprocessing first')

df = pd.read_csv(data_path, parse_dates=['datetime']).sort_values('datetime').set_index('datetime')
df = df[~df.index.duplicated(keep='first')]

print(df.shape)
print(df.head())


(9357, 16)
                       CO(GT)  PT08.S1(CO)  NMHC(GT)  C6H6(GT)  PT08.S2(NMHC)  \
datetime                                                                        
2004-03-10 18:00:00  0.345514     1.029801 -0.402597  0.057638       0.221365   
2004-03-10 19:00:00 -0.073785     0.739028 -0.630636 -0.202266      -0.064850   
2004-03-10 20:00:00  0.065981     1.209396 -0.774660 -0.243850      -0.115173   
2004-03-10 21:00:00  0.065981     1.098218 -0.822668 -0.223058      -0.086866   
2004-03-10 22:00:00 -0.353318     0.653506 -0.996697 -0.503753      -0.439130   

                      NOx(GT)  PT08.S3(NOx)   NO2(GT)  PT08.S4(NO2)  \
datetime                                                              
2004-03-10 18:00:00 -0.354188      0.898330  0.065624      0.487077   
2004-03-10 19:00:00 -0.663469      1.357292 -0.391267      0.152876   
2004-03-10 20:00:00 -0.526011      1.225048  0.087381      0.142824   
2004-03-10 21:00:00 -0.324732      1.038352  0.261435      0.21569

Define columns and generate lag/rolling features.

In [2]:
pollutant_cols = ['CO(GT)', 'NMHC(GT)', 'C6H6(GT)', 'NOx(GT)', 'NO2(GT)']
sensor_cols = ['PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)', 'PT08.S4(NO2)', 'PT08.S5(O3)']
weather_cols = ['T', 'RH', 'AH']

pollutant_cols = [c for c in pollutant_cols if c in df.columns]
sensor_cols = [c for c in sensor_cols if c in df.columns]
weather_cols = [c for c in weather_cols if c in df.columns]

print('Pollutant columns:', pollutant_cols)
print('Sensor columns:', sensor_cols)
print('Weather columns:', weather_cols)

max_lag = 24

for col in pollutant_cols + sensor_cols + weather_cols:
    for lag in [1, 6, 12, 24]:
        df[f"{col}_lag_{lag}"] = df[col].shift(lag)

for col in pollutant_cols + sensor_cols + weather_cols:
    for window in [6, 12, 24]:
        df[f"{col}_rolling_mean_{window}"] = df[col].shift(1).rolling(window=window, min_periods=1).mean()

# Drop rows with missing due to lag/rolling at the start
engineered_df = df.dropna()
print('Engineered shape:', engineered_df.shape)


Pollutant columns: ['CO(GT)', 'NMHC(GT)', 'C6H6(GT)', 'NOx(GT)', 'NO2(GT)']
Sensor columns: ['PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)', 'PT08.S4(NO2)', 'PT08.S5(O3)']
Weather columns: ['T', 'RH', 'AH']
Engineered shape: (9333, 107)


fe_path = artifacts / 'feature_engineered.csv'
engineered_df.to_csv(fe_path)
print(f'Saved engineered features to {fe_path}')


In [3]:
fe_path = artifacts / 'feature_engineered.csv'
engineered_df.to_csv(fe_path)
print(f'Saved engineered features to {fe_path}')


Saved engineered features to artifacts\feature_engineered.csv
